In [ ]:
%pip install spacy

In [ ]:
%pip install scispacy

In [1]:
#imports
import spacy
import scispacy
import timeit
import pandas as pd
import os
from pathlib import Path
from collections import defaultdict
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from collections import Counter
import plotly.express as px


/home/exouser/Desktop/bridge-rna/.venv-1/lib/python3.12/site-packages/torch/cuda/__init__.py:188: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /__w/pytorch/pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
/home/exouser/Desktop/bridge-rna/.venv-1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#load term parser
nlp = spacy.load("en_core_sci_lg")

In [49]:
#change this to live file ingest later
vectorizer = CountVectorizer(lowercase=True,)
file_path = Path.cwd().parent / "archs4metadata/COHORT|study=OSD-100|spaceflight=Ground Control_hits.csv"

#md_df = pd.read_parquet(file_path) if file_path.suffix == ".parquet" else pd.read_csv(file_path)
md_df = pd.read_csv(file_path)

idx = defaultdict(set) #may want to cache later if it gets big

In [7]:
def _add_spacy_terms(): # adds spacy terms to dataframe, returns list of flat terms
    md_df['spacey_terms'] = md_df.apply(lambda _: [], axis=1)

    for row in md_df.itertuples():
        doc = nlp(row.geo_summary)
        for ent in doc.ents:
            term = ent.text.split()
            row.spacey_terms.extend(term)
            for t in term:
                idx[t].add(row.gse)
            
    
    return [item for lst in md_df["spacey_terms"] for item in lst]

In [8]:
def _update_idx():
    
    analyzer = vectorizer.build_analyzer()
    new_idx = defaultdict(set)

    for term,doc_ids in idx.items():
        tokenized = analyzer(term)
        for token in tokenized:
            new_idx[token].update(doc_ids)

    return new_idx  


In [9]:
def _add_journals(cl_df):
     cl_df['journals'] = cl_df.apply(lambda _: [], axis=1)
     cl_df["Representation"] = cl_df["Representation"].apply(lambda lst: [x for x in lst if x])

     for row in cl_df.itertuples():
        
        
        for term in row.Representation:
            
            if term in idx:
               row.journals.extend(list(idx[term]))
            else:
                  print(f"term {term} not found in index")
     
     return cl_df      
         

In [10]:

def cluster_sample(): 
    global idx
    flat_terms = _add_spacy_terms()
    vectorizer.fit(flat_terms)
    idx = _update_idx()
    model = SentenceTransformer('allenai/biomed_roberta_base')
    topic_model = BERTopic(vectorizer_model=vectorizer, embedding_model=model)
    topics,probs = topic_model.fit_transform(flat_terms)
    clusters = _add_journals(topic_model.get_topic_info())   
    return topic_model,clusters
    


In [11]:
def get_cluster_distribution(clusters,cluster_idx):
    journals = clusters.iloc[cluster_idx].journals
    cnts = Counter(journals)
   
    total = len(journals)

    percent_dist =  {name: (count / total) * 100 for name, count in cnts.items()}
    return percent_dist

In [12]:
def get_cluster_counts(clusters,cluster_idx):
    journals = clusters.iloc[cluster_idx].journals
    cnts = Counter(journals)
    cnts = dict(cnts)
    return cnts

In [14]:
def get_top_cluster_distribution(clusters,n=5):
    top_clusters = clusters.head(n)
    distributions = {}
    for idx, row in top_clusters.iterrows():
        distributions[row.Topic] = get_cluster_distribution(clusters,idx)
    return distributions

In [15]:
def get_top_journals(clusters,n=5):
    
    top_clusters = clusters.head(n).iloc[1:] #skip outliers
    overall_cnts = defaultdict(int)
    for idx, row in top_clusters.iterrows():
       # print(f"Processing cluster {row.Topic} at index {idx}")
        journal_counts = get_cluster_counts(clusters,idx)
       # print(f"Journal counts for cluster {row.Topic}: {journal_counts}")
        for journal, count in journal_counts.items():
            #print(f"Adding {count} to overall count for journal {journal}")
            overall_cnts[journal] += count
    
    return overall_cnts

In [16]:
def get_top_journal_distribution(clusters):
    top_cnt_dict = get_top_journals(clusters)
    total = sum(top_cnt_dict.values())
    percent_dist = {name: (count / total) * 100 for name, count in top_cnt_dict.items()}
    return percent_dist


In [50]:
model,clusters = cluster_sample() #47s no gpu on VM 

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 23224.53it/s]
[transformers] RobertaModel LOAD REPORT from: allenai/biomed_roberta_base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [51]:
model.get_topic_info().head(10)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,32,-1_disease_expressed_modifier_macrophages,"[disease, expressed, modifier, macrophages, re...","[disease-associated, disease-resistant, diseas..."
1,0,69,0_eye_female_aged_loss,"[eye, female, aged, loss, process, function, l...","[eye, eye, eye]"
2,1,41,1_genes_eyecups_removal_coincidental,"[genes, eyecups, removal, coincidental, corpse...","[genes, genes, genes]"
3,2,33,2_mertk_r345w_doyne_ngs,"[mertk, r345w, doyne, ngs, kos, wheeler, rsem4...","[Mertk, Mertk, Mertk]"
4,3,32,3_retina_retinal_posterior_receptor,"[retina, retinal, posterior, receptor, pro, ph...","[retinal, retina, retina]"
5,4,31,4_analysis_degeneration_study_anova,"[analysis, degeneration, study, anova, workflo...","[analysis, analysis, analysis]"
6,5,29,5_tyro3_tug1_ki_efemp1ki,"[tyro3, tug1, ki, efemp1ki, efemp1, tyrosine, ...","[Efemp1ki/ki, Efemp1ki/ki, Efemp1ki/ki]"
7,6,27,6_expression_mouse_changes_cells,"[expression, mouse, changes, cells, cell, canc...","[expression, expression, mouse]"
8,7,26,7_biological_malattia_dystrophy_phenotypes,"[biological, malattia, dystrophy, phenotypes, ...","[dystrophy/malattia, dystrophy/malattia, dystr..."
9,8,24,8_data_set_models_model,"[data, set, models, model, role, values, syste...","[data, data, data]"


In [52]:
top_j = get_top_journals(clusters)
jour_dict = get_top_journal_distribution(clusters)
jour_dist = pd.DataFrame(list(get_top_journal_distribution(clusters).items()),columns=['Journal','Percentage'])

In [53]:
jour_dist

,Journal,Percentage
0,GSE106591,20.454545
1,GSE210492,20.454545
2,GSE124745,9.090909
3,GSE205070,45.454545
4,GSE189618,2.272727
5,GSE143281,2.272727


In [54]:
pd.DataFrame(get_top_cluster_distribution(clusters)) #note: includes outliers as -1 index

,-1,0,1,2,3
GSE205070,81.818182,58.333333,50.000000,20.0,50.0
GSE106591,9.090909,8.333333,25.000000,40.0,10.0
GSE143281,9.090909,NaN,NaN,NaN,10.0
GSE210492,NaN,25.000000,8.333333,20.0,30.0
GSE124745,NaN,8.333333,8.333333,20.0,NaN
GSE189618,NaN,NaN,8.333333,NaN,NaN


In [ ]:
list(jour_dict.keys())

In [ ]:
list(jour_dict.items())

In [45]:
def _plot_dist(dist):
    fig = px.pie(
    names = list(jour_dict.keys()),
    values = list(jour_dict.values()),
   
    title="Percentage Breakdown of Journals",
    )

    fig.show(renderer="browser")


In [46]:
_plot_dist(jour_dist)

In [37]:
def save_cluster_data(clusters, filename="cluster_data.csv"):
    clusters.to_csv(filename, index=False)

In [55]:
save_cluster_data(clusters, filename="COHORT|study=OSD-100|spaceflight=Ground Control_top10hits_clusters.csv")

In [57]:
def compare_ground_spaceflight(clusters_ground,clusters_spaceflight):
    #see how much the clusters overlap between the two datasets
    #show common clusters, unique clusters, and distribution of journals in each cluster
    

    #create a set of intersection of clusters
    common_clusters = set(clusters_ground["Name"]).intersection(set(clusters_spaceflight["Name"]))
    unique_clusters_ground = set(clusters_ground["Name"]).difference(set(clusters_spaceflight["Name"]))
    unique_clusters_spaceflight = set(clusters_spaceflight["Name"]).difference(set(clusters_ground["Name"]))

    common_journals =   set(clusters_ground["journals"]).intersection(set(clusters_spaceflight["journals"]))
    unique_journals1 = set(clusters_ground["journals"]).difference(set(clusters_spaceflight["journals"]))
    unique_journals2 = set(clusters_spaceflight["journals"]).difference(set(clusters_ground["journals"]))

    #create a dataframe with the information
    comparison_df =pd.DataFrame({
        "common_clusters": list(common_clusters),
        "unique_clusters_ground_control": list(unique_clusters_ground),
        "unique_clusters_spaceflight": list(unique_clusters_spaceflight),
        "common_journals": list(common_journals),
        "unique_journals_ground_control": list(unique_journals1),
        "unique_journals_spaceflight": list(unique_journals2)
    })
   
    
                     
                                                



    return comparison_df

In [61]:
!ls

'COHORT|study=OSD-100|spaceflight=Ground_Control_top10hits_clusters.csv'
'COHORT|study=OSD-100|spaceflight=Space_Flight_top10hits_clusters.csv'
 cluster_metadata.ipynb
 requirements.txt


In [64]:
ground_file_path = 'clustering/COHORT|study=OSD-100|spaceflight=Ground_Control_top10hits_clusters.csv' 
spaceflight_file_path = "clustering/COHORT|study=OSD-100|spaceflight=Spaceflight_top10hits_clusters.csv"

#full paths to the files
full_ground_file_path = Path.cwd().parent / ground_file_path
full_spaceflight_file_path = Path.cwd().parent / spaceflight_file_path

ground_df = pd.read_csv(ground_file_path)
spaceflight_df = pd.read_csv(full_spaceflight_file_path)

compare_df = compare_ground_spaceflight(ground_df, spaceflight_df)  

FileNotFoundError: [Errno 2] No such file or directory: 'clustering/COHORT|study=OSD-100|spaceflight=Ground_Control_top10hits_clusters.csv'

In [ ]:
#visualize the comparison
